# Example SFINCS Run — Standard

Run a staged Single-Use-Case example against the base model built by the matching `a_build_*` notebook.

> **Stage Contract**
>
> Requires: standard SFINCS base model and a sample event forcing window
>
> Produces: single example standard run outputs for smoke review
>
> Next: 05_create_scenarios.ipynb for production scenarios

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import os
os.environ.pop("DEBUG", None)

from hydromt_sfincs import SfincsModel, DATADIR
import hydromt

print("hydromt_sfincs:", __import__('hydromt_sfincs').__version__)
print("hydromt:", hydromt.__version__)


In [ ]:
import sys
from pathlib import Path

location_root = Path("../..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))
workspace = location_root

from sfincs_runs.notebook import load_runtime

runtime = load_runtime(location_root)
config = runtime.config
paths = runtime.paths

static_dir = runtime.static_dir
sfincs_root = runtime.sfincs_root
base_model = runtime.base_model

design_outputs = runtime.design_outputs
events_dir = runtime.events_dir
dep_dir = runtime.dep_dir
catalog_dir = runtime.catalog_dir

# Local rasters.
cudem_tif = static_dir / "cudem_mfield_mesh.tif"   # CUDEM clipped to the imported mesh footprint, EPSG:26919
worldcover_tif = static_dir / "worldcover_mfield_mesh.tif"
mesh_hdf = None
mesh_clean = static_dir / "unstructured_mfield_mesh_ugrid.nc"

# Mapping table bundled with hydromt_sfincs.
esa_mapping = Path(DATADIR) / "lulc" / "esa_worldcover_mapping.csv"

print("Location workspace:", workspace)
print("Base model root:", base_model)


## Rerun Control


In [ ]:
rerun = False


## Single-Use-Case Test

In [ ]:
# Single-Use-Case event forcing from the Event Catalog built in 03_build_event_catalog.ipynb.
from sfincs_runs.scenarios.event_forcing import build_event

SINGLE_USE_EVENT_ID = "evt_0003"
ZSINI_MODE = "dry"
HUTHRESH = 0.01

single_use_event = build_event(
    config,
    paths,
    event_id=SINGLE_USE_EVENT_ID,
    zsini_mode=ZSINI_MODE,
)
single_use_case_plan = single_use_event.plan
event_forcing = single_use_event.forcing
base_model = single_use_event.base_model_root
pd.DataFrame(single_use_case_plan.summary_rows())


In [ ]:
# Inspect the catalog-selected boundary water level and paired forcing metadata.
from sfincs_runs.scenarios.event_forcing import hydrology_inputs
from sfincs_runs.scenarios.io import parse_sfincs_inp

hydrology_cfg = (config.get("coastal_wave_coupling") or {}).get("hydrology") or {}
infiltration_cfg = hydrology_cfg.get("infiltration") or {}
hydrology_enabled = bool(infiltration_cfg.get("enabled", True))
hydrologic_drivers = (
    set(config.get("event_drivers") or []) & {"rainfall", "soil_moisture"}
    if hydrology_enabled
    else set()
)
include_precip = bool(hydrologic_drivers)

h_series = event_forcing.h
t_start = event_forcing.t_start
t_stop = event_forcing.t_stop
zsini = event_forcing.zsini
base_inp = parse_sfincs_inp(base_model / "sfincs.inp")
EVENT_LABEL = event_forcing.event_id
hydrology_inputs = (
    hydrology_inputs(event_forcing, paths=paths, config=config)
    if include_precip
    else None
)

print(f"Event: {event_forcing.event_id}")
print(f"WL peak: {float(h_series.max()):.2f} m MSL")
print(f"zsini: {zsini:.4f} m ({ZSINI_MODE})")
print("Rain/soil drivers:", "enabled" if include_precip else "disabled")
if hydrology_inputs:
    print("Rainfall source:", Path(hydrology_inputs["rainfall_source_nc"]).name)
    print("Soil wetness variable:", hydrology_inputs["soil_moisture_summary"].get("soil_moisture_variable", ""))

fig, ax = plt.subplots(figsize=(10, 3.5))
h_series.plot(ax=ax, color="#0b6e99", linewidth=2)
ax.axhline(zsini, color="#d95f02", linestyle="--", linewidth=1.5, label=f"zsini = {zsini:.2f} m")
ax.set_title(f"{EVENT_LABEL} boundary water level")
ax.set_ylabel("Water level (m MSL)")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Stage the Event-Catalog forcing. This standard notebook is surge/rainfall only.
from sfincs_runs.scenarios import audit_forcing
from sfincs_runs.scenarios.event_forcing import run_model, stage_precip, stage_run

figures_dir = single_use_case_plan.storage_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

staged = stage_run(
    base_model,
    single_use_case_plan.run_root,
    event_forcing,
    force=rerun,
    include_waves=False,
    include_precip=include_precip,
    timing_config={"allow_legacy_inference": True},
)
run_root = staged.run_root
staged_manifest = staged.manifest
# Reload the base model so this example notebook can run standalone after the build notebook.
sf = SfincsModel(root=str(base_model), mode="r+")
precip_manifest = None
if include_precip:
    precip_manifest = stage_precip(
        sf,
        run_root,
        event_forcing,
        paths=paths,
        config=config,
    )
    staged_manifest.update(precip_manifest)
    print("Precipitation forcing:", Path(precip_manifest["prepared_precip"]).name)

forcing_audit = audit_forcing(run_root)
if forcing_audit.issues:
    print("Forcing audit:")
    for issue in forcing_audit.issues:
        print(f"  [{issue.severity}] {issue.code}: {issue.message}")
if not forcing_audit.passed:
    raise RuntimeError("Forcing audit failed: " + ", ".join(sorted(forcing_audit.error_codes)))

run_start = pd.Timestamp(staged_manifest["run_start"])
run_stop = pd.Timestamp(staged_manifest["run_stop"])
print("Staged run:", run_root)
print(f"SFINCS support window: {run_start} to {run_stop} ({staged_manifest['run_duration_hours']:.1f} h)")
print("Timing policy:", staged_manifest["timing_policy"])
print(f"Boundary water-level rows: {len(h_series)}")


### Step 10c · Forcing QA plots

Plot the staged forcing files before running SFINCS so the water level, rainfall, soil/infiltration state, and manifest metadata can be checked on the staged SFINCS clock.


In [ ]:
from sfincs_runs.diagnostics import plot_standard_forcing

plot_standard_forcing(
    run_root=run_root,
    out_dir=figures_dir / "forcing",
    event_id=single_use_case_plan.event_id,
    event_label=EVENT_LABEL,
    h_series=h_series,
    staged_manifest=staged_manifest,
    precip_manifest=precip_manifest,
    include_precip=include_precip,
    hydrology_inputs=hydrology_inputs,
)


In [ ]:
# Run SFINCS after the staged forcing QA checks.
run_result = run_model(run_root, config=config)
run_map = run_result.map_path
print("Run completed:", run_map)


In [ ]:
from sfincs_runs.diagnostics import plot_standard_animation
from IPython.display import HTML, Video

out_mp4 = plot_standard_animation(
    run_root=run_root,
    out_dir=figures_dir,
    event_id=single_use_case_plan.event_id,
    event_label=EVENT_LABEL,
    h_series=h_series,
    t_start=t_start,
    zsini=zsini,
    huthresh=HUTHRESH,
)
display(HTML(f"<h4>{EVENT_LABEL} — flood + ocean animation</h4>"))
display(Video(str(out_mp4), embed=True))


### Post-run diagnostics · soil moisture · runup gauge · precip

In [ ]:
from sfincs_runs.diagnostics import plot_standard_diagnostics

plot_standard_diagnostics(
    run_root=run_root,
    event_label=EVENT_LABEL,
    t_start=t_start,
    hydrology_inputs=hydrology_inputs,
    has_rugfile=False,
)


### Post-run diagnostics · AORC precipitation animation

In [ ]:
from sfincs_runs.diagnostics import plot_precip_animation
from IPython.display import HTML, Video

out_mp4 = plot_precip_animation(
    run_root=run_root,
    out_dir=figures_dir,
    event_id=single_use_case_plan.event_id,
    event_label=EVENT_LABEL,
)
if out_mp4:
    display(HTML(f"<h4>{EVENT_LABEL} — AORC precipitation animation</h4>"))
    display(Video(str(out_mp4), embed=True))
